In [1]:
import warnings
warnings.filterwarnings('ignore')

from IPython.core.display import display, HTML
display(HTML('<style>.container {width: 90% !important; }</style>'))

import glob, pyBigWig, shutil, os
import pandas as pd

import sys
sys.path.append('crestgv/')
from crestgv import crestgv

import seaborn as sns
import matplotlib.pyplot as plt

## Functions to calculate "null" enrichment scores and p-values

In [20]:
def run_null_cgv(genetic, output, random_collection="", random_collection_path=""):
    # Run CREST-GV to get the enrichment scores
    cgv_null = crestgv(genetic=genetic, output=output, 
                       min_number_genetics=100,
                       collection_name=random_collection, in_house_collection_path=random_collection_path) 
    df_cgv_null = cgv_null.calculate_enrichment_score(lessNG=True, greater25k=True) 
    return 

def run_bootstrap(output, N_BOOTS, SEED): 
    # Read the intermediate scores to generate the null distribution
    rounds_null = sorted(glob.glob(output + "/rounds/statistics_intermediate_round*")) 
    dfs_null = []
    for r in rounds_null:
        df = pd.read_csv(r, sep='\t', index_col=0)
        df = df[['CREST-GV']]
        dfs_null.append(df)
        
    df_null = pd.concat(dfs_null)

    # Sample with replacement 10,000 times from df_null to generate a full null
    df_null_bootstrap = df_null.sample(n=N_BOOTS, replace=True, random_state=SEED)
    
    return df_null_bootstrap

def generate_pvalue_normalised_score(genetic, output, real_df_cgv, random_collection="", random_collection_path="", N_BOOTS=10000, SEED=42):
    # Get the null enrichment scores
    run_null_cgv(genetic = genetic, output = output, 
                 random_collection=random_collection, random_collection_path=random_collection_path)
    # Get the null bootstrap distribution (this will plot a figure as it runs)
    df_null_bootstrap = run_bootstrap(output = output, N_BOOTS=N_BOOTS, SEED=SEED)
    
    # to get the normalised score, divide the enrichment score by the mean of the empirical null
    df_cgv_with_pvals = real_df_cgv
    emp_mean = df_null_bootstrap['CREST-GV'].mean()
    df_cgv_with_pvals['normalised_score'] = real_df_cgv['CREST-GV_all'].apply(lambda x: (x / emp_mean))

    # for each real enrichment score count the number of times a value equal to or greater than the observed value was sampled in the bootstrap null and divide by the number of samples
    df_cgv_with_pvals['empirical_pvalue'] = real_df_cgv['CREST-GV_all'].apply(lambda x: (df_null_bootstrap['CREST-GV'] >= x).sum()/N_BOOTS)  

    return df_cgv_with_pvals

## Calculate enrichment scores for a range of genetics, super-PBMC

In [16]:
# Get the list of genetic files
gfiles = glob.glob("/project/Wellcome_Discovery/svenkat/CREST-GV/data/normalise_scores_test/*.txt")
# Remove file folders and extensions
gfile_bases = [os.path.splitext(os.path.basename(file))[0] for file in gfiles]
gfile_bases

['09_Waldenstrom_macroglobulinemia',
 '06_leukocyteDisease',
 '03_MS',
 '10_CysticFibrosis',
 '13_IBD',
 '08_gout',
 '00_Astle',
 '05_bodyheight',
 '07_parkinsonDisease',
 '11_bipolarDisorder',
 '01_type1diabetes',
 '12_schizophrenia',
 '02_type2diabetes',
 '04_intelligence']

In [ ]:
for gname in gfile_bases:
    genetic = "/project/Wellcome_Discovery/svenkat/CREST-GV/data/normalise_scores_test/" + gname + ".txt"
    
    # Get the un-normalised scores
    output = "normalise_scores_test/" + gname + "_super_pbmc/"
    cgv = crestgv(genetic = genetic, 
                  output = output, min_number_genetics=100, collection_name = "super_pbmc")
    df_cgv = cgv.calculate_enrichment_score(lessNG=True, greater25k=True)
    
    # Get the "random" cell type scores for p-values and normalising
    random_collection_path = "/project/Wellcome_Discovery/svenkat/CREST-GV/data/perc_10"
    output = "normalise_scores_test/" + gname + "_random_perc_10/"

    df_cgv_with_pvals = generate_pvalue(genetic = genetic, output = output, real_df_cgv = df_cgv,
                                        random_collection = "", random_collection_path = random_collection_path)
    df_cgv_with_pvals = df_cgv_with_pvals.sort_values('CREST-GV_all', ascending = False)
    
    res_file = "normalise_scores_test/" + gname + "_results.txt"
    df_cgv_with_pvals.to_csv(res_file)

Removing NaN rows from loaded file ...
Total number of used variants in CREST-GV: 3
Calculating (1) total number of base-pairs within peaks and (2) total number of base-pairs within peaks divided by uniquely mappable base-pairs ...


100%|██████████| 19/19 [00:00<00:00, 23.55it/s]


Intersecting genetic with peak regions ...


100%|██████████| 19/19 [00:04<00:00,  4.40it/s]

For each fold, parallelised shuffling background ...
